# EJEMPLO DE PROYECTO
# CLASIFICACIÓN DE PACIENTES CON ANEMIA
# DATASET : DATOS ABIERTOS PERÚ

In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [2]:
!pip install pyjanitor
import janitor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 4.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
DATASET_URL = 'https://raw.githubusercontent.com/cesarmayta/datasets/refs/heads/main/TB_ANEMIA.csv'
df_anemia = pd.read_csv(DATASET_URL,sep=';')
df_anemia

,id_persona,Edad,Tipo_edad,Sexo,id_ubigeo,Fecha_atencion,Diagnostico,Tipo_Dx,id_eess
0,36178991,6,M,F,NaN,20210524,D509,D,2137
1,38986407,2,A,M,1128.0,20211118,D509,D,10836
2,39002414,1,A,F,1128.0,20210417,D509,D,10836
3,39002414,1,A,F,1128.0,20210413,D509,D,10836
4,39002414,3,A,F,1128.0,20220612,D509,D,10836
...,...,...,...,...,...,...,...,...,...
1327919,40158681,6,M,M,1288.0,20220308,D509,D,5998
1327920,38778349,11,A,F,1320.0,20220319,D529,D,16380
1327921,41045581,5,A,F,1309.0,20230428,D509,D,6085
1327922,38603474,11,A,M,1315.0,20220803,D649,D,6216


# EDA

In [8]:
df_anemia.shape

(1327924, 9)

In [9]:
df_anemia.dtypes

,0
id_persona,int64
Edad,int64
Tipo_edad,object
Sexo,object
id_ubigeo,float64
Fecha_atencion,int64
Diagnostico,object
Tipo_Dx,object
id_eess,int64


# TRATAMIENTO DE NULOS

In [10]:
df_anemia.isna().sum()

,0
id_persona,0
Edad,0
Tipo_edad,0
Sexo,0
id_ubigeo,10817
Fecha_atencion,0
Diagnostico,0
Tipo_Dx,0
id_eess,0


In [11]:
df_anemia.dropna(subset=['id_ubigeo'],inplace=True)
df_anemia.shape

(1317107, 9)

# TRATAMIENTO DE DUPLICADOS

In [12]:
df_anemia.duplicated().sum()

np.int64(19287)

In [13]:
df_anemia[df_anemia.duplicated()]

,id_persona,Edad,Tipo_edad,Sexo,id_ubigeo,Fecha_atencion,Diagnostico,Tipo_Dx,id_eess
16,39347963,1,A,F,1128.0,20210814,D509,D,21206
28,41907576,6,M,F,1209.0,20240316,D509,D,17076
574,39320800,3,A,F,1504.0,20210312,D509,D,87
629,38701922,4,A,M,42.0,20220104,D509,D,5180
660,39320673,4,A,F,42.0,20221029,D509,D,5188
...,...,...,...,...,...,...,...,...,...
1327161,42009830,6,M,M,1453.0,20240507,D509,D,12
1327223,41940907,7,M,M,1000.0,20240613,D509,D,3356
1327377,42037152,6,M,F,1276.0,20240518,D509,D,4404
1327466,42068631,6,M,M,1322.0,20240530,D509,D,5846


In [14]:
df_anemia.drop_duplicates(keep='first',inplace=True)
df_anemia.shape

(1297820, 9)

# ELIMINAR COLUMNAS QUE NO ME SERVIRAN PARA EL ENTRENAMIENTO DEL MODELO

In [15]:
df_anemia.drop(['id_persona','Fecha_atencion','id_eess'],axis=1,inplace=True)

In [16]:
df_anemia

,Edad,Tipo_edad,Sexo,id_ubigeo,Diagnostico,Tipo_Dx
1,2,A,M,1128.0,D509,D
2,1,A,F,1128.0,D509,D
3,1,A,F,1128.0,D509,D
4,3,A,F,1128.0,D509,D
5,3,A,F,1542.0,D509,D
...,...,...,...,...,...,...
1327918,1,A,F,1316.0,D519,D
1327919,6,M,M,1288.0,D509,D
1327920,11,A,F,1320.0,D529,D
1327921,5,A,F,1309.0,D509,D


In [17]:
df_anemia['cie10'] = np.where(df_anemia['Diagnostico'].str.startswith(('D50','D53','D64')),1,0)

In [20]:
df_anemia['edad_total'] = np.where(
    df_anemia['Tipo_edad'] == 'M', df_anemia['Edad'] / 12,
    np.where(df_anemia['Tipo_edad'] == 'D', df_anemia['Edad'] / 365, df_anemia['Edad'])
)

In [21]:
df_anemia

,Edad,Tipo_edad,Sexo,id_ubigeo,Diagnostico,Tipo_Dx,cie10,edad_total
1,2,A,M,1128.0,D509,D,1,2.0
2,1,A,F,1128.0,D509,D,1,1.0
3,1,A,F,1128.0,D509,D,1,1.0
4,3,A,F,1128.0,D509,D,1,3.0
5,3,A,F,1542.0,D509,D,1,3.0
...,...,...,...,...,...,...,...,...
1327918,1,A,F,1316.0,D519,D,0,1.0
1327919,6,M,M,1288.0,D509,D,1,0.5
1327920,11,A,F,1320.0,D529,D,0,11.0
1327921,5,A,F,1309.0,D509,D,1,5.0


In [22]:
df_anemia_clean = df_anemia[['edad_total','cie10','Sexo']].copy()
df_anemia_clean.rename(columns={'Sexo':'sexo','edad_total':'edad','cie10':'anemia'},inplace=True)

In [24]:
df_anemia_clean.dtypes

,0
edad,float64
anemia,int64
sexo,object


In [ ]:
df_anemia_clean = pd.get_dummies(df_anemia_clean, columns=['sexo'], drop_first=True).astype(int)

In [29]:
df_anemia_clean['sexo_M'] = df_anemia_clean['sexo_M'].astype(int)

In [31]:
df_anemia_clean.rename(columns={'Sexo_M':'sexo'},inplace=True)

In [32]:
df_anemia_clean

,edad,anemia,sexo_M
1,2.0,1,1
2,1.0,1,0
3,1.0,1,0
4,3.0,1,0
5,3.0,1,0
...,...,...,...
1327918,1.0,0,0
1327919,0.5,1,1
1327920,11.0,0,0
1327921,5.0,1,0


In [34]:
df_anemia_clean.to_csv('anemia_clean.csv',index=False,encoding='utf-8')